# Motex V3（模型本体）

自清洁的 Decoder-only 语言模型，作为 Motex 系列的新版本（v1/v2/v2_1/v2_2 之后）。

## 相比前代的改进
1. **内置因果注意力掩码**：训练/预填充/增量生成在结构上自洽，不依赖外部 `valid_lens` 的传法保证因果性。
2. 标准组合：Pre-Norm(RMSNorm) + GQA(RoPE + KV-Cache) + SwiGLU FFN + 权重绑定。
3. 统一返回 `(logits, state, aux_loss=0.0)`，兼容 `motex_utils.training` 的共享训练/预测接口。
4. 模型实现收敛在 `motex_utils/motex_v3.py`。

> 本 notebook 只展示**模型本体**的构建与使用（它是仓库主题）；
> 真实数据加载与训练管线（分词器/语料/多轮训练）在仓库 `dev/` 目录（不入库）。

In [ ]:
# 仓库根目录加入 sys.path（本仓库自包含）
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import torch
from torch import nn

from motex_utils.motex_v3 import MotexV3, generate

## 模型构建与前向接口

`MotexV3(vocab_size, d_model, num_layers, num_heads, num_kv_heads, ffn_hidden, dropout, max_seq_len)`

统一接口：`forward(tokens, valid_lens=None, state=None) -> (logits, state, aux_loss)`。

In [ ]:
net = MotexV3(vocab_size=4096, d_model=256, num_layers=4, num_heads=4,
              num_kv_heads=2, ffn_hidden=512, dropout=0.1, max_seq_len=128)
print('参数(M):', round(sum(p.numel() for p in net.parameters()) / 1e6, 2))

# 训练模式前向
x = torch.randint(5, 4096, (2, 128))
logits, state, aux = net(x, None, None)
print('train 前向 logits', tuple(logits.shape), ' 返回 3 元组，aux=', aux.item())

## 生成演示（内置迷你字符分词器，仅演示模型接口）

真实训练使用 BPE/字符分词器在 `dev/` 的管线中构建；此处用最小字符映射做自包含演示。

In [ ]:
class MiniTok:
    def __init__(self, chars):
        self.stoi = {s: i for i, s in enumerate(['<pad>', '<unk>', '<bos>', '<eos>', '\n'] + chars)}
        self.itos = {v: k for k, v in self.stoi.items()}
    def encode(self, t): return [self.stoi.get(c, 1) for c in t]
    def decode(self, ids): return ''.join(self.itos.get(i, '?') for i in ids)

# 用一个含常用字的字符集重建模型（与上面同接口）
chars_used = list('你我他她说目的地方向山里大人心有是这让可也看要走回么了')
tok = MiniTok(chars_used)
net2 = MotexV3(vocab_size=len(tok.stoi), d_model=256, num_layers=4, num_heads=4,
               num_kv_heads=2, ffn_hidden=512, dropout=0.1, max_seq_len=128)
net2.eval()
text, _ = generate(net2, tok, '你', 40, 'cpu', temperature=0.9, top_k=20,
                   repetition_penalty=1.1)
print('（未训练模型，输出为随机字符；展示 KV-Cache 生成接口可用）')
print(text)

## 说明
- 训练循环复用 `motex_utils.training.train_motex_ckpt_v2`（与 v1/v2 一致）；
- 在真实语料上从零训练并达到流畅成句的实验（数据/分词/多变体对比/断点）见仓库 `dev/` 目录。